### This file contains the synthesis process for the dataset we'll use in the classification process later

In [49]:
## libraries for loading data
import pandas as pd
import wfdb
from pathlib import Path
from scipy.signal import butter, filtfilt, welch

In [27]:
## libraries for data parsing
import numpy as np
import neurokit2 as nk # ECG peak detection, HRV metrics
import heartpy as hp # Pulse/PPG analysis (PLETH signal)
from scipy.stats import entropy # Signal statistics & peak detection

In [29]:
## libraries for parallel feature execution
from concurrent.futures import ProcessPoolExecutor

We'll aim to create a dataset with 19 features for each record. To start off, we'll start with some functions to determine signal quality: Signal-to-noise Ratio, Baseline Wander, and Spectral Entropy.

1. Signal-to-noise ratio helps as low SNR means the ECG waveform is dominated by noise, which often triggers false alarms.
2. Baseline wander is the amount of slow drift in the ECG baseline. This is useful since baseline drift occurs when electrodes loosen or when the patient moves, both of which can cause false alarms.
3. Spectral Entropy is the entropy of the signal’s frequency spectrum, with a high level of entropy indicating radndom noise.

In [32]:
# Signal quality functions
def signal_snr(signal):
    signal_power = np.mean(signal**2)
    noise_power = np.var(signal - np.mean(signal))
    return signal_power / noise_power

def baseline_wander(signal, fs):
    b, a = butter(2, 0.5/(fs/2), btype="low")
    baseline = filtfilt(b, a, signal)
    return np.std(baseline)

def spectral_entropy(signal, fs):
    f, Pxx = welch(signal, fs)
    Pxx_norm = Pxx / np.sum(Pxx)
    return entropy(Pxx_norm)

Now, we'll go through each of the columns in the record and descern several features from each, starting with the 'II' column.

The ECG Statistical Features describe the basic distribution and amplitude of the ECG waveform, other ECG features capture heartbeat timing and rhythm patterns.

Here's a little synopsis about the usefulness of each ECG metric:
1. ECG mean is useful since a large offset can indicate baseline drift or electrode problems, which often cause false alarms.
2. ECG standard deviation can help distinguish between real rhythm changes and noise spikes.
4. ECG min/max is useful as false VTach alarms frequently come from abnormally large spikes.
5. ECG amplitude strenth helps as very large ranges may indicate artifact rather than physiological activity.
6. ECG peak count and RR mean measures the number of detected heart-beats, which matters as VTech is consistent with rapid heartbeats.
7. RR standard deviation helps detech arrythmias, but can also be influenced by noise.
8. RR coeficient of variation (calculated by RR mean / RR std) is used to measure consistency in intervals, which may indicate noise.
9. Heart Rate Variability Standard deviation of NN (normal-to-normal) intervals (hrv_sdnn) is used to measure overall heartrate vairability, as low variability may occur during sustained tachycardia.
10. Heart Rate Variability Root mean square of successive RR differences (hrv_rmssd) is used to help distinguish structured rhythm patterns with random peak detections caused by noise.

In [35]:
def extract_ecg_features(ecg, fs):

    features = {}

    features["ecg_mean"] = np.mean(ecg)
    features["ecg_std"] = np.std(ecg)
    features["ecg_max"] = np.max(ecg)
    features["ecg_min"] = np.min(ecg)
    features["ecg_amplitude_range"] = np.max(ecg) - np.min(ecg)

    features["ecg_snr"] = signal_snr(ecg)
    features["baseline_wander"] = baseline_wander(ecg, fs)
    features["spectral_entropy"] = spectral_entropy(ecg, fs)

    try:
        signals, info = nk.ecg_process(ecg, sampling_rate=fs)
        peaks = info["ECG_R_Peaks"]

        features["ecg_peak_count"] = len(peaks)

        if len(peaks) > 1:
            rr = np.diff(peaks) / fs

            features["rr_mean"] = np.mean(rr)
            features["rr_std"] = np.std(rr)
            features["rr_cv"] = np.std(rr) / np.mean(rr)

            hrv = nk.hrv_time(peaks, sampling_rate=fs)

            features["hrv_sdnn"] = hrv["HRV_SDNN"].values[0]
            features["hrv_rmssd"] = hrv["HRV_RMSSD"].values[0]

        else:
            features["rr_mean"] = np.nan
            features["rr_std"] = np.nan
            features["rr_cv"] = np.nan
            features["hrv_sdnn"] = np.nan
            features["hrv_rmssd"] = np.nan

    except:
        features["ecg_peak_count"] = np.nan
        features["rr_mean"] = np.nan
        features["rr_std"] = np.nan
        features["rr_cv"] = np.nan
        features["hrv_sdnn"] = np.nan
        features["hrv_rmssd"] = np.nan

    return features

Below is the code to help extract the Plethysmography (PLETH) Features from each record.

Uses for each PLETH metric:
1. PLETH mean can be helpful as extremely low or flat values may indicate poor sensor contact or signal loss.
2. PLETH standard deviation can also help detect noise as low variation may indicate sensor issues or motion artifact.
3. PLETH bpm represents the actual blood pulse rate, which, when different from ECG bpm, could mean a false alarm.
4. PLETH also measures heart rate variability calculated from the pleth signal (pleth_sdnn), which provides a second estimate of rhythm variability, allowing cross-signal validation.
5. PLETH peak count helps estimate pulse frequency and confirm whether blood flow matches ECG rhythm.

In [38]:
def extract_pleth_features(pleth, fs):

    features = {}

    features["pleth_mean"] = np.mean(pleth)
    features["pleth_std"] = np.std(pleth)

    try:
        wd, m = hp.process(pleth, fs)

        features["pleth_bpm"] = m["bpm"]
        features["pleth_sdnn"] = m["sdnn"]
        features["pleth_peak_count"] = len(wd["peaklist"])

    except:
        features["pleth_bpm"] = np.nan
        features["pleth_sdnn"] = np.nan
        features["pleth_peak_count"] = np.nan

    return features

We also have some cross-signal feautures that take into account different columns from the waveform data:
1. HR difference is the difference between ECG heartrate and PLETH heartrate, which can indicate noise or improper placement of instruments.
2. We also track the correlation between the ECG and PLETH data (ecg_pleth_corr), which gives us yet another way to quanitfy the amount of noise in the data

In [41]:
def extract_cross_signal_features(ecg, pleth, ecg_features, pleth_features):

    features = {}

    try:
        ecg_hr = 60 / ecg_features["rr_mean"]
        pleth_hr = pleth_features["pleth_bpm"]

        features["hr_difference"] = abs(ecg_hr - pleth_hr)

    except:
        features["hr_difference"] = np.nan

    try:
        corr = np.corrcoef(ecg, pleth)[0,1]
        features["ecg_pleth_corr"] = corr

    except:
        features["ecg_pleth_corr"] = np.nan

    return features

Now that we have methods for calculating the different features, we can code a loop that calculates all of the metrics for all of the different waveform records. Testing showed that there were a small amount of the records that didn't have all of the columns, which is why we search for column name instead of normal index.

In [53]:
def extract_features(signals, fs, sig_names):

    features = {}

    # ECG (assume first signal if not labeled)
    ecg = signals[:, 0]
    ecg_features = extract_ecg_features(ecg, fs)
    features.update(ecg_features)

    # Pleth may not exist
    pleth = None
    if "PLETH" in sig_names:
        pleth_index = sig_names.index("PLETH")
        pleth = signals[:, pleth_index]

        pleth_features = extract_pleth_features(pleth, fs)
        features.update(pleth_features)

        # Cross-signal features only if both exist
        cross_features = extract_cross_signal_features(
            ecg, pleth, ecg_features, pleth_features
        )
        features.update(cross_features)

    else:
        # Optional: fill expected pleth features with NaN
        features["pleth_missing"] = 1

    return features

Now we can (finally!) apply this to our actual records and set up our dataframe.

In [58]:
root_dir = Path(r"C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms")

rows = []

for hea in root_dir.rglob("*.hea"):

    record_name = hea.stem
    record_path = hea.with_suffix('')  # removes .hea
    print(record_path)

    record = wfdb.rdrecord(str(record_path))

    signals = record.p_signal
    fs = record.fs
    sig_names = record.sig_name   # <-- add this line

    features = extract_features(signals, fs, sig_names)
    features["record"] = record_name

    rows.append(features)

df = pd.DataFrame(rows)

C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\003c13\003c13_0115
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\003c13\003c13_0126
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004bad\004bad_0015


C:\Users\gocle\anaconda3\Lib\site-packages\neurokit2\ecg\ecg_clean.py:106: NeuroKitWarning: There are 87576 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004bad\004bad_1115
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004bad\004bad_1426
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004bad\004bad_2348
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004f0c\004f0c_2475
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004f0c\004f0c_2533
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004f0c\004f0c_3642
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004f0c\004f0c_3644
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\004fdb\004fdb_0105
C:\Users\gocle\Downloads\Ventricular Tachycardia

C:\Users\gocle\anaconda3\Lib\site-packages\numpy\ma\core.py:5334: RuntimeWarning: Mean of empty slice.
  result = super().mean(axis=axis, dtype=dtype, **kwargs)[()]
C:\Users\gocle\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3787: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\0179fd\0179fd_2773
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\0183b7\0183b7_0219
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\01a952\01a952_1067
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\01a952\01a952_2485
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\01a952\01a952_2516
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\01a952\01a952_3291
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\01d569\01d569_0333
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\01e98d\01e98d_0143
C:\Users\gocle\Downloads\Ventricular Tachycardia

...

C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe46f9\fe46f9_0194
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe817b\fe817b_2246
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe87a5\fe87a5_1001
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe9bf2\fe9bf2_0070
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe9f80\fe9f80_0120
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe9f80\fe9f80_0241
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe9f80\fe9f80_0322
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\fe9f80\fe9f80_0458
C:\Users\gocle\Downloads\Ventricular Tachycardia

C:\Users\gocle\anaconda3\Lib\site-packages\neurokit2\ecg\ecg_clean.py:106: NeuroKitWarning: There are 2087 missing data points in your signal. Filling missing values using `signal_fillmissing`.
  warn(


C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff7587\ff7587_0096
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff7587\ff7587_0099
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff7587\ff7587_0104
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff7587\ff7587_0109
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff7587\ff7587_0121
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff8c49\ff8c49_0107
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff8c49\ff8c49_0114
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ff8c49\ff8c49_0117
C:\Users\gocle\Downloads\Ventricular Tachycardia

C:\Users\gocle\anaconda3\Lib\site-packages\heartpy\analysis.py:677: UserWarning: 
The maximal number of iterations maxit (set to 20 by the program)
allowed for finding a smoothing spline with fp=s has been reached: s
too small.
There is an approximation returned but the corresponding weighted sum
of squared residuals does not satisfy the condition abs(fp-s)/s < tol.
  interp = UnivariateSpline(x, rrlist, k=3)


C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ffecba\ffecba_0687
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ffecba\ffecba_0740
C:\Users\gocle\Downloads\Ventricular Tachycardia Alarm Classification Project\data1\data2\waveforms\ffecba\ffecba_0838


In [61]:
df

,ecg_mean,ecg_std,ecg_max,ecg_min,ecg_amplitude_range,ecg_snr,baseline_wander,spectral_entropy,ecg_peak_count,rr_mean,...,hrv_rmssd,pleth_mean,pleth_std,pleth_bpm,pleth_sdnn,pleth_peak_count,hr_difference,ecg_pleth_corr,record,pleth_missing
0,-0.003995,0.176943,3.080022,-0.76000,3.840022,1.000510,0.042344,3.051028,658.0,0.546801,...,187.967304,0.495905,0.115011,118.228279,95.385744,655.0,8.499067,-0.078850,003c13_0115,NaN
1,-0.003853,0.187560,1.094985,-0.83000,1.924985,1.000422,0.043749,2.987488,656.0,0.547823,...,204.255881,0.493117,0.130811,130.983338,125.401451,716.0,21.458892,-0.083184,003c13_0126,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,0.658769,...,187.161250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,004bad_0015,NaN
3,-0.011857,0.509036,7.150031,-5.48500,12.635031,1.000543,0.474469,2.537037,498.0,0.721400,...,9.578201,0.446772,0.146346,83.323132,8.767140,427.0,0.151567,-0.004110,004bad_1115,NaN
4,-0.000483,0.174927,1.035014,-1.21000,2.245014,1.000008,0.144078,3.571050,554.0,0.649013,...,144.882161,0.449594,0.143425,86.682608,86.102088,524.0,5.765512,-0.065623,004bad_1426,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5028,0.023974,0.131832,1.090666,-1.06872,2.159386,1.033071,0.010028,3.359583,574.0,0.626862,...,126.005217,4.670070,1.647256,99.102719,105.229746,633.0,3.387892,0.088728,ffe78d_0899,NaN
5029,0.025106,0.139872,1.205359,-0.74664,1.952000,1.032219,0.007914,3.224269,541.0,0.665356,...,65.100693,4.682066,1.713606,90.320427,65.136102,581.0,0.143078,0.021338,ffe78d_1182,NaN
5030,0.020936,0.190926,0.595009,-1.04000,1.635009,1.012024,0.134476,2.206652,398.0,0.901108,...,422.626755,0.501311,0.172670,74.702314,49.122447,460.0,8.117644,-0.275589,ffecba_0687,NaN
5031,0.013798,0.149244,0.804994,-1.37500,2.179994,1.008547,0.069722,2.978735,414.0,0.867119,...,108.252133,0.506115,0.173566,69.747227,41.666012,440.0,0.552543,-0.020298,ffecba_0740,NaN


In [63]:
df.to_csv("records_features.csv")